# Session 9: From RNN to LSTM to Transformer
**Instructor:** Soham Bhattacharya &nbsp;|&nbsp; **Date:** 01-07-2026

> **How to use these notes:** Each section contains the original lecture content, then a clearly marked **📚 Additional Learning** block. Lecture content = what was taught in class. Additional Learning = deeper explanation, beginner intuition, common mistakes, best practices, and references. The two are always separated so you know the source.

## Table of Contents

1. [The Evolution of Sequence Models](#1-evolution)
2. [The Vanishing Gradient Problem](#2-vanishing)
3. [LSTM — Long Short-Term Memory](#3-lstm)
4. [The Four Gates of LSTM](#4-gates)
5. [Token-by-Token Walkthrough](#5-walkthrough)
6. [LSTM Architecture in Code](#6-code)
7. [Transformer — Brief Introduction](#7-transformer)
8. [Reference: Session 8 Hidden State Simulation](#8-session8)
9. [Quick Reference](#9-reference)

---
<a id="1-evolution"></a>
## 1. The Evolution of Sequence Models

The session opened with a recap of the architectural progression the course has been tracing:

```
Bag-of-Words → RNN → LSTM → Attention → Transformer → LLMs
```

**Previous session recap (from slide):**
- Bag-of-words ignored word order entirely
- RNN solved order by using a hidden state
- RNN reads one token at a time
- RNN struggles with long-term dependencies

**This session covers:**
- LSTM improves memory
- Attention removes the memory bottleneck
- Transformer uses self-attention
- Modern LLMs build on the Transformer architecture

### 📚 Additional Learning — Section 1

**Why this progression matters**

Each step in that chain exists because the previous approach had a specific, concrete failure. Bag-of-words couldn't tell "dog bites man" from "man bites dog." RNNs fixed word order but forgot things they read early in a long sentence. LSTMs fixed forgetting but still processed words one at a time in sequence, which is slow to train. Attention let the model look at all words simultaneously. Transformers built attention into a scalable architecture. LLMs are Transformers trained at massive scale.

Understanding *why* each successor exists makes the architecture choices feel inevitable rather than arbitrary.

**Further reading**
- Andrej Karpathy's *"The Unreasonable Effectiveness of Recurrent Neural Networks"* — [karpathy.github.io/2015/05/21/rnn-effectiveness/](http://karpathy.github.io/2015/05/21/rnn-effectiveness/) — accessible motivation for why sequence models matter, written before Transformers dominated.
- *"Attention Is All You Need"* — [arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762) — the 2017 paper that introduced the Transformer. Reading the abstract and introduction gives you a feel for what problem it was solving.

---
<a id="2-vanishing"></a>
## 2. The Vanishing Gradient Problem

### Definition
As sequence context grows, early inputs fade. Mathematical gradients shrink exponentially when backpropagating through long unrolled loops.

### Explanation
The vanishing gradient problem is why standard RNNs fail on long sentences. The three failure modes the slide identifies are:

- **Short-term context:** The RNN can handle adjacent modulations (e.g., "not good") because those tokens are close together.
- **Long-term decay:** Early words lose their impact over dozens of steps. By the time the model reaches the end of a long sentence, the signal from the beginning has largely disappeared.
- **Vanishing gradient:** Early context is forgotten during training updates because the gradient signal shrinks toward zero as it travels backward through time.

### Example (from slide diagram)
The classic example shown: the word "France" appears early in a sentence, but by the time the model needs to predict "French" many tokens later, the hidden state no longer retains a strong signal from "France."

### ⚠️ Key Takeaway (from slide)
Standard RNNs struggle with long sentences because backpropagating gradients shrink toward zero.

### 📚 Additional Learning — Section 2

**Beginner intuition**

Imagine passing a telephone message down a line of 50 people. Each person whispers to the next. By the time the message reaches person 50, it has degraded. An RNN has the same problem: the signal about what the model read at position 1 gets multiplied by a weight matrix at every subsequent step. If that weight is slightly less than 1 (common), multiplying it by itself 50 times gives a number approaching zero. The gradient — the learning signal telling the model how to improve — vanishes before it reaches the early layers.

The reverse problem (exploding gradients) also exists: if the weight is slightly greater than 1, repeated multiplication produces very large numbers that destabilize training. Gradient clipping is the standard fix for exploding gradients, but clipping cannot recover a signal that has already vanished.

**Why backpropagation through time specifically causes this**

RNNs are trained by "unrolling" them — conceptually, you copy the same network once per time step and apply standard backpropagation through the unrolled chain. A 50-word sentence becomes a 50-layer network for gradient computation. Gradients that need to travel from step 50 back to step 1 traverse all 50 layers, experiencing multiplicative shrinkage at every step.

**Common mistake**

Beginners often confuse vanishing gradients with the model "forgetting" at inference time. They are related but distinct problems. Vanishing gradients are a *training* problem — the model cannot learn long-range dependencies because the learning signal does not reach the weights responsible for early tokens. Even if you could freeze a perfectly trained RNN, it would still struggle at inference on very long sequences because its hidden state has finite capacity and overwrites itself as new tokens arrive. LSTM addresses both issues.

**Further reading**
- *"On the difficulty of training recurrent neural networks"* — Pascanu et al., 2013 — [arxiv.org/abs/1211.5063](https://arxiv.org/abs/1211.5063)
- Keras documentation on recurrent layers: [keras.io/api/layers/recurrent_layers/](https://keras.io/api/layers/recurrent_layers/)

---
<a id="3-lstm"></a>
## 3. LSTM — Long Short-Term Memory

### Definition
LSTM stands for Long Short-Term Memory. It is an RNN with better memory control.

**Formula (from lecture):**
```
LSTM = RNN + CELL STATE + GATES
```

### Explanation
LSTMs keep memory safe by using specialized gates to protect and manage information along a dedicated cell state pathway. The core idea, stated directly in the lecture slide:

> *Decide what to keep, what to remove, what to add, and what to expose.*

An LSTM adds six things to a standard RNN:

| Addition | Role |
|---|---|
| Cell state | Long-term memory; travels through the sentence |
| Hidden state | Short-term memory; current working memory |
| Forget gate | Removes outdated information |
| Input gate | Writes new information to memory |
| Candidate memory | Proposed values to write into the cell state |
| Output gate | Selects what to expose from the cell state |

The cell state is described as a **conveyor belt** — a long-term memory path that travels through the sentence. The hidden state is the current working memory at each step.

### The Two States (from the live notebook)

In [ ]:
#Cell state in LSTM is long term memory
#hidden state in lstm is short term memory

#cell state like notbook that travels through the sentence

### RNN vs LSTM (from interactive visualization at engineersofai.com)

| | RNN | LSTM |
|---|---|---|
| Memory | Single hidden state h_t | Hidden state h_t + separate cell state c_t |
| Gradient flow | Vanishes over long sequences | Flows through c_t without squashing |
| Long sequences | Fails after ~10–20 steps | Enables 100s of time steps |

### 📚 Additional Learning — Section 3

**A concrete analogy for the two states**

Think of a student attending a week-long course. The **cell state** is their notebook — they selectively write things down, cross things out, and carry it forward each day. The **hidden state** is their working memory at any given moment — what they are actively thinking about as they listen to the current sentence. When each lecture ends (a time step passes), they update the notebook (cell state) based on what they just heard, and they start the next session with a fresh working focus (hidden state) informed by everything in the notebook.

The critical insight is that the notebook has a near-direct path through time. It does not pass through the same squashing functions as the hidden state at every step, which is why gradients can flow through it without vanishing.

**Why the cell state solves the vanishing gradient problem**

In a standard RNN, information must travel through the hidden state at every step, and every step applies a `tanh` activation that squashes values into [-1, 1]. Multiplying these squashed values repeatedly causes the gradient to vanish. The LSTM cell state instead travels through an *additive* update path:

```
c_new = f * c_prev + i * g
```

where `f`, `i`, and `g` are gate values. This additive structure means gradients can flow backward through the `f * c_prev` term without being repeatedly squashed, as long as the forget gate `f` stays close to 1 for the relevant information.

**Common mistake: confusing cell state and hidden state**

This came up directly in the live session (a student asked: "is cell state, current working memory?"). The answer is no. Cell state = long-term memory. Hidden state = current working memory. When you call `lstm_model.predict()`, the output is derived from the **hidden state**, not directly from the cell state. The cell state is internal bookkeeping that the gates use to regulate what enters the hidden state.

**Original paper**

LSTM was introduced by Sepp Hochreiter and Jürgen Schmidhuber in 1997: *"Long Short-Term Memory,"* Neural Computation, 9(8), 1735-1780. Available at: [deeplearning.cs.cmu.edu/F23/document/readings/LSTM.pdf](https://deeplearning.cs.cmu.edu/F23/document/readings/LSTM.pdf)

**Further reading**
- Christopher Olah's *"Understanding LSTM Networks"* — [colah.github.io/posts/2015-08-Understanding-LSTMs/](http://colah.github.io/posts/2015-08-Understanding-LSTMs/) — the most-cited visual walkthrough of LSTM internals.
- TensorFlow LSTM documentation: [tensorflow.org/api_docs/python/tf/keras/layers/LSTM](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM)

---
<a id="4-gates"></a>
## 4. The Four Gates of LSTM

### Definition
Gates are mechanisms inside an LSTM that control what information is kept, discarded, added, or exposed at each time step.

### The Gates (from the "LSTM Gates" slide)

**Forget gate:** What old information should be removed?  
**Input gate & Candidate:** What new information should be added/stored?  
**Output gate:** What should be exposed now?

### Mathematical Formulas (from the interactive visualization)

```
Forget: f = σ(Wf[h,x]+b)
Input:  i = σ(Wi[h,x]+b)
Cell:   g = tanh(Wg[h,x]+b)
Output: o = σ(Wo[h,x]+b)
```

### Visual Diagram (from Edureka, shown in lecture)

```
              Input Gate

  Forget Gate   [1]  [2]  [3]   Output Gate

Forget Irreverent               Pass Updated
  Information → [LSTM] ←       Information

            Add / Update New
              Information
```

Forget Gate removes irrelevant information. Steps 1–3 inside the block handle the sequential operations. The output gate passes updated information out.

### Interactive Visualization
The instructor demonstrated this live at **[engineersofai.com/playground/lstm-gates](https://engineersofai.com/playground/lstm-gates)**

Three adjustable inputs:
- `INPUT X_T (x)` — the current input token value
- `HIDDEN H_PREV (h)` — the previous hidden state
- `CELL STATE C_PREV (c)` — the previous cell state

Gate bar chart labels:
- Forget: *"much of old memory to keep (0=forget, 1=keep)"*
- Input: *"How much new info to write to memory"*
- Cell: *"Candidate values to add to memory (tanh)"*
- Output: *"How much of memory to expose as output"*

**Live demonstration result:** `c_prev=-0.50 → c_new=-0.41`, gate values: `f=0.5348, i=0.3647, g=-0.3826, o=0.4008`

### ⚠️ Important Points
- The cell state acts as a memory highway, shielding early context from vanishing gradients.
- **Cell state ≠ current working memory.** The hidden state is current working memory. The cell state is long-term memory.
- The forget gate *drops* irrelevant information; it does not keep it.

### 📚 Additional Learning — Section 4

**Reading the gate formulas**

Every gate formula shares the structure `σ(W[h,x]+b)`:

- `[h,x]` — concatenate the previous hidden state and the current input into one vector. The gate looks at both what just happened and what is being said right now before deciding what to do.
- `W` — a weight matrix learned during training. You do not set this manually.
- `b` — a bias term, also learned.
- `σ` — the sigmoid function, squashing any number to [0, 1]. This makes the gate output interpretable as a "percentage of information to let through."

The Cell gate uses `tanh` instead of `σ` because it is computing a candidate value (which can be negative or positive, in [-1, 1]), not a gate percentage.

**How to read the visualization numbers**

`f=0.5348` means the forget gate is keeping about 53% of the previous cell state. `i=0.3647` means the input gate is writing about 36% of the candidate value into memory. These values are not manually tuned — they emerge from learned weight matrices applied to the current x and h. When you "Randomize Weights," you are simulating what different trained weights would produce.

**Worked example: forget gate in the lecture walkthrough**

At token 5 ("Like"), the instructor noted `forget_gate-the`. The word "The" (stored from token 1) is being dropped. "The" served its purpose as an article — it signalled that a noun phrase was starting — but it carries no semantic weight going forward. The forget gate's value for "The"-related memory would be close to 0 at this step, effectively zeroing it out of the cell state.

At token 6 ("LSTM"), the forget gate drops "does" — an auxiliary verb that helped establish verb structure but is no longer needed once the object (LSTM) arrives.

**Common mistake: thinking gates are manually programmed**

The gate values are *not* rule-based. There is no code that says "forget adjectives after the noun arrives." All gate weights are learned from data during training via backpropagation. The walkthrough in the lecture is a *conceptual* interpretation — it teaches you to think like the model, not describing hard-coded rules.

**Best practice: use the interactive tool**

Recommended exercise: check "Compare RNN vs LSTM" in the visualization, set a high forget gate value, press Compute Step repeatedly, and observe how the cell state highway propagates information across many steps without collapsing. Then uncheck the comparison and observe that the RNN's hidden state decays much faster.

**Further reading**
- TensorFlow guide to recurrent neural networks: [tensorflow.org/guide/keras/working_with_rnns](https://www.tensorflow.org/guide/keras/working_with_rnns)
- *"An Empirical Exploration of Recurrent Network Architectures"* — Jozefowicz et al., 2015 — [proceedings.mlr.press/v37/jozefowicz15.pdf](http://proceedings.mlr.press/v37/jozefowicz15.pdf). Found that the forget gate is the single most important LSTM component; initializing its bias to 1 (strongly "keep by default") consistently improved results.

---
<a id="5-walkthrough"></a>
## 5. Token-by-Token Walkthrough — LSTM Processing a Sentence

The main live coding exercise. The instructor traced how an LSTM's hidden state and cell state evolve token by token through:

**"The class does not like LSTM even though it is good, and LLMS are built on an extended version of ____"**

**Task:** predict the next word — the expected answer is "Transformer".

In [ ]:
#the class does not like LSTM even though it is good, and LLMS are built on an extended version of ____
#Predict the next word

#Token 1- The
#hidden state-sentence started

#Token 2- Class
#hidden state- subject is class

#Token 3- does
#hidden state- waiting for description

#Token 4- not
#hidden state- negation activated

#token 5- Like
#hidden state- sentiment word like found, indicating to be good but core meaning/ memory does not change
#forget_gate-the

#token 6- LSTM
#hidden state- update object- LSTM- class not like lstm
#forget gate- drops does

#token 7-even
#hideen state- waiting for for context

#token 8- though
#hidden state- wating for more info

#token 9- it
#hidden_state- poining to lstm

#token 10- is
#hidden state - waiting for info

#token 11- good
#forget_gate-even though it
#input_gate-class not like lstm, lstm is good
#candidate state- lstm is good

#token 12-and
#hidden state- more info required

#token 13-LLMS
#hidden state- added new subject LLMS
#input gate- store llms

#token 14-


#last token-
#output-gate- 1. LSTM, 2.class

### What this demonstrates

- The **hidden state** tracks what the model is currently focused on. It updates with every token.
- The **forget gate** actively drops tokens when they are no longer needed (token 5 drops "the"; token 6 drops "does").
- The **input gate** stores new subjects (token 13 stores "LLMS").
- The **cell state** carries long-term information — "class does not like lstm" persists all the way to the final prediction.
- The **output gate** at the final token exposes the two candidate answers: 1. LSTM, 2. class. The model is expected to pick "Transformer" as the extended version.

*Student Nancy GUPTA correctly identified that the forget gate for token 11 drops "even though it."*

### 📚 Additional Learning — Section 5

**Why this exercise is pedagogically important**

Most LSTM tutorials show the architecture diagram and math, then immediately jump to training code. This walkthrough forces you to *reason like a model*, tracing information through the gates manually. That mental model is what lets you debug LSTM behavior in practice. If your model keeps ignoring negation ("not good" being classified as positive), you can now hypothesize that the hidden state is failing to carry `negation_active=True` forward.

**Try it yourself**

Repeat this exercise with: *"The restaurant was not very bad but the service was terrible."* Trace:
- What does the hidden state carry after "not"?
- What does the forget gate drop after "but"? (Hint: contrast often signals a perspective shift.)
- What should the candidate state store at "terrible"?
- What would the output gate expose at the end?

There is no single correct answer — the point is developing the habit of thinking about information flow through time.

**Connection to the code**

Each "hidden state" entry in the walkthrough corresponds to the 16-dimensional vector `h_t` that `layers.LSTM(16)` outputs at each step. The scalar descriptions ("negation activated") are human-interpretable summaries of what might be encoded in those 16 numbers. In a trained model, those 16 dimensions contain a distributed, non-interpretable encoding — but the *logic* of what gets retained and dropped follows exactly the pattern the instructor traced.

**Common mistake: assuming the LSTM always "gets it right"**

The output gate at the last token shows "1. LSTM, 2. class" — the model correctly identifies LSTM as highly relevant. But whether it would actually predict "Transformer" as the completion depends entirely on whether "Transformer" appeared as a completion to similar sentences in training data. LSTMs learn statistical patterns, not logical reasoning. A model trained only on cooking blogs would have no idea what "Transformer" means in this context.

---
<a id="6-code"></a>
## 6. LSTM Architecture in Code

### What not to do — the syntax error from the live session

The instructor's first attempt wrote both imports on a single line:

In [ ]:
#simple lstm architecture via code

# ❌ This causes a SyntaxError — do not do this
import tensorflow as tf from tensorflow.keras import layers

SyntaxError: invalid syntax

### Corrected LSTM Architecture (from the live notebook)

In [ ]:
#simple lstm architecture via code

import tensorflow as tf
from tensorflow.keras import layers

vocab_size = 1000
embedding_dim = 16
sequence_length = 8

lstm_model = tf.keras.Sequential([
    layers.Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        input_length=sequence_length
    ),
    layers.LSTM(16),#dimensionality || size of the hidden state
    layers.Dense(1, activation="sigmoid")
])

lstm_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┌─────────────────────────────┬──────────────────┬───────────────┐
│ Layer (type)                │ Output Shape     │ Param #       │
├─────────────────────────────┼──────────────────┼───────────────┤
│ embedding (Embedding)       │ ?                │ 0 (unbuilt)   │
│ lstm (LSTM)                 │ ?                │ 0 (unbuilt)   │
│ dense (Dense)               │ ?                │ 0 (unbuilt)   │
└─────────────────────────────┴──────────────────┴───────────────┘
 Total params: 0 (0.00 B)
 Trainable params: 0 (0.00 B)
 Non-trainable params: 0 (0.00 B)


### Layer Explanations (from notebook and data flow diagram)

**Embedding Layer**  
Converts token IDs to dense vectors.  
- `input_dim=vocab_size` — vocabulary size: 1,000 words  
- `output_dim=embedding_dim` — each word becomes a 16-dimensional vector  
- Output shape: `(batch_size, 8, 16)` — 8 time steps, each a 16-d vector

**LSTM Layer**  
Reads the sequence of vectors step by step with better memory control.  
- Processes all 8 steps internally  
- Outputs only the **final** summary hidden state  
- Output shape: `(batch_size, 16)` — the dimensionality is the size of the hidden state  
- The code comment: `#dimensionality || size of the hidden state`

**Dense Layer**  
Makes the final prediction.  
- Multiplies 16 inputs by learned weights, adds bias  
- `activation="sigmoid"` squashes the output to a single number between 0 and 1

**For sentiment classification:**  
`1 = positive` | `0 = negative`

### Data Flow (from the instructor's text diagram)

```
[Input: 8 words]
    |
    ▼
[Embedding Layer] ——→ Output shape: (batch_size, 8, 16)
    |
    ▼
[LSTM Layer]      ——→ Processes 8 steps internally.
                       Outputs final summary state only.
    |——→ Output shape: (batch_size, 32)   ← Note: diagram was written before
    |                                        instructor changed LSTM(32) to LSTM(16)
    ▼
[Dense Layer]     ——→ Multiplies inputs by weights, adds bias.
    |
    ▼
[Sigmoid]         ——→ Squashes output to a single number between 0 and 1.
```

### ⚠️ Important Points
- `input_length` is deprecated in the Keras version used (TF/Python 3.12). The warning is harmless but the argument should be removed in new code.
- All params show `0 (unbuilt)` because `summary()` was called before the model saw any data. This is expected behaviour, not an error.
- The `16` in `LSTM(16)` is the hidden state dimensionality, not the number of LSTM cells in sequence — the single layer already handles all 8 time steps internally.

### Layer annotation cell (from notebook)

In [ ]:
from numpy import negative
#Embedding layer:
    #token-token ids-vectors

#LSTM Layer:
    #Reads sequence of vectors step by step with better memory control

#Dense Layer:
    #Makes the final prediction
    #sigmoid- Gives value between 0 and 1

#sentiment based lstm
1 postive
0 negative

#prediction based

### 📚 Additional Learning — Section 6

**Understanding each hyperparameter**

`vocab_size = 1000` — the model knows 1,000 distinct words. Every word must be an integer ID from 0 to 999. Unknown words are mapped to a special `<UNK>` token in production. GPT-2 uses a vocabulary of 50,257 — 1,000 is intentionally small for learning the architecture.

`embedding_dim = 16` — each word is represented as a 16-dimensional vector. Production models typically use 128, 256, or 768. The embedding layer *learns* these vectors during training: words with similar meanings end up with similar vectors, even though nothing was programmed to enforce that.

`sequence_length = 8` — every input is exactly 8 tokens. Real sentences vary in length. You handle this by padding shorter sentences with `<PAD>` and truncating longer ones. Keras provides `tf.keras.preprocessing.sequence.pad_sequences()` for this.

`layers.LSTM(16)` — the 16 is the number of hidden units. Larger = more expressive but slower to train and more prone to overfitting on small datasets.

**Fixing the deprecation warning**

Simply remove `input_length=sequence_length` from the Embedding layer:

```python
layers.Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim
    # input_length removed — no longer needed
)
```

**Seeing parameter counts before training**

Call `lstm_model.build()` before `summary()`:

```python
lstm_model.build(input_shape=(None, sequence_length))
lstm_model.summary()
```

This forces Keras to resolve shapes and will print real parameter counts.

**Note on the data flow diagram**

The diagram lists LSTM output shape as `(batch_size, 32)` and "Multiplies 32 inputs." This reflects an earlier version of the code where `LSTM(32)` was used. The instructor revised it to `LSTM(16)` before saving. The correct output shape for the final code is `(batch_size, 16)`.

**Common mistakes**

1. **`return_sequences=True` when you only want one label per sentence.** The default `return_sequences=False` returns only the final hidden state. Setting it `True` returns h_t at every time step — useful for sequence-to-sequence tasks, but will break the Dense layer shape here unless you add `layers.Flatten()` or `layers.GlobalAveragePooling1D()`.

2. **`activation="softmax"` for binary classification.** Softmax is for multi-class (N outputs that sum to 1). For binary, use `sigmoid` on a single output unit.

3. **Forgetting to compile before training.** `summary()` works without compiling, but `model.fit()` requires `model.compile(optimizer=..., loss=..., metrics=...)`.

**Best practice: compile call for binary sentiment classification**

```python
lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
```

Use `binary_crossentropy` because the output is a single sigmoid probability. Use `adam` as the default optimizer — it requires no manual learning rate tuning to get started.

**Official documentation**
- [`tf.keras.layers.Embedding`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding)
- [`tf.keras.layers.LSTM`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/LSTM)
- [`tf.keras.layers.Dense`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense)
- [`tf.keras.Sequential`](https://www.tensorflow.org/api_docs/python/tf/keras/Sequential)

**Further reading**
- TensorFlow text classification tutorial: [tensorflow.org/text/tutorials/text_classification_rnn](https://www.tensorflow.org/text/tutorials/text_classification_rnn) — complete end-to-end example on IMDB sentiment using this exact architecture.
- Keras RNN guide: [keras.io/guides/working_with_rnns/](https://keras.io/guides/working_with_rnns/) — covers masking, stateful RNNs, and bidirectional LSTMs.

---
<a id="7-transformer"></a>
## 7. Transformer — Brief Introduction

The Transformer Explainer tool ([poloclub.github.io/transformer-explainer](https://poloclub.github.io/transformer-explainer)) was shown briefly.

**Definition shown on screen:**

> *Transformer is the core architecture behind modern AI, powering models like ChatGPT and Gemini. Introduced in 2017, it revolutionized how AI processes information. The same architecture is used for training on massive datasets and for inference to generate outputs. Here we use GPT-2 (small), simpler than newer ones but perfect for learning the fundamentals.*

**What the visual showed:**
- A Transformer Block containing: Embedding, Multi-head Self Attention, MLP
- 12 identical Transformer Blocks stacked
- Output probabilities for next-token prediction (example: "visualize" at 54.67%, "create" at 20.87%)
- Key, Query, Value matrices in the attention mechanism
- Head 1 of 12 (multi-head attention)

This was a preview only — the instructor did not go deep on Transformers in this session.

```
Bag-of-Words → RNN → LSTM → Attention → Transformer → LLMs
                              ↑
                         Current focus
```

### 📚 Additional Learning — Section 7

**Why Transformers replaced LSTMs**

LSTMs process tokens *sequentially* — token 5 cannot be processed until tokens 1 through 4 are done. This prevents parallelisation across a sentence, which makes LSTMs slow to train on large datasets. Transformers process all tokens in a sequence *simultaneously* using attention, making them vastly faster to train on modern GPUs. The tradeoff is that Transformers require more memory (attention scales quadratically with sequence length), but hardware improvements have largely addressed this.

The probability output in the visualization — "visualize" at 54.67%, "create" at 20.87% — directly illustrates how a language model works: it produces a probability distribution over the entire vocabulary at every position, and the next token is sampled from that distribution.

**What Key, Query, Value means (preview)**

The attention mechanism's Key/Query/Value structure will be covered in a future session. The intuition: for every word in the sentence, Query asks "what am I looking for?", Key says "here is what I contain," and Value says "here is what I will contribute if you attend to me." The dot product of Query and Key determines how much attention one word pays to another.

**Getting ahead: explore the tool**

The Transformer Explainer at [poloclub.github.io/transformer-explainer](https://poloclub.github.io/transformer-explainer) is interactive. You can type your own prompt and watch the model assign attention weights between words in real time. Exploring it before the next session will make the formal treatment much easier to follow.

**Further reading**
- *"The Illustrated Transformer"* by Jay Alammar — [jalammar.github.io/illustrated-transformer/](http://jalammar.github.io/illustrated-transformer/) — the standard beginner-friendly visual walkthrough.
- *"Attention Is All You Need"* — [arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762) — the original Transformer paper. The architecture section (Section 3) is readable without deep math background.

---
<a id="8-session8"></a>
## 8. Reference: Session 8 Hidden State Simulation

The instructor briefly opened `b1_s8.ipynb` to show a related example: a hand-written simulation of an RNN-style hidden state tracking sentiment through "The movie was not good."

This was shown to connect the intuitive token-by-token trace in Session 9 back to the code approach from Session 8.

In [ ]:
if token in ['good', 'bad', 'great', 'terrible']:
    hidden_state['sentiment_word'] = token

print("Token:", token)
print("Hidden state:", hidden_state)
print("------")

Token: the
Hidden state: {'topic': None, 'negation_active': False, 'sentiment_word': None}
------
Token: movie
Hidden state: {'topic': 'movie', 'negation_active': False, 'sentiment_word': None}
------
Token: was
Hidden state: {'topic': 'movie', 'negation_active': False, 'sentiment_word': None}
------
Token: not
Hidden state: {'topic': 'movie', 'negation_active': True, 'sentiment_word': None}
------
Token: good
Hidden state: {'topic': 'movie', 'negation_active': True, 'sentiment_word': 'good'}
------


The hidden state dictionary approach from Session 8 maps directly onto the informal token-by-token trace the instructor built in this session's notebook.

### 📚 Additional Learning — Section 8

**What this code was actually demonstrating**

`b1_s8.ipynb` is a hand-written *simulation* of what a hidden state should track — not a neural network. It uses explicit `if` statements to update a dictionary. The conceptual operations are identical to an LSTM (track topic, track negation, track sentiment), but in a real LSTM those operations are *learned from data automatically* rather than hard-coded.

**The limitation this code exposes**

Notice that `hidden_state['sentiment_word']` gets set to `'good'` even though `negation_active` is `True`. The code stores the sentiment word but does not combine it with the negation flag to produce a final sentiment. A real trained LSTM would learn to combine these signals — the cell state would carry both pieces of information and the final hidden state would reflect their interaction. This is exactly the gap the Session 9 architecture code begins to fill.

---
<a id="9-reference"></a>
## 9. Quick Reference

### Key Terms

| Term | Lecture definition |
|---|---|
| Vanishing gradient | Gradients shrink exponentially when backpropagating through long unrolled loops |
| LSTM | RNN + Cell State + Gates |
| Cell state | Long-term memory; conveyor belt that travels through the sentence |
| Hidden state | Short-term memory; current working memory |
| Forget gate | Decides what old information to remove |
| Input gate | Decides what new information to add |
| Candidate memory | The proposed new values to write into the cell state |
| Output gate | Decides what to expose from the cell state as the current output |
| Embedding layer | Converts token IDs to vectors |
| sigmoid | Squashes output to a value between 0 and 1 |

### Resources from the Session

| Resource | URL |
|---|---|
| LSTM Gates interactive visualization | engineersofai.com/playground/lstm-gates |
| Transformer Explainer | poloclub.github.io/transformer-explainer |
| Session 9 notebook | colab.research.google.com/drive/1IQlmH-gUIPsMShMFa-IG7vTzPUleM0D2 |

### Official Documentation

| | |
|---|---|
| `tf.keras.layers.LSTM` | tensorflow.org/api_docs/python/tf/keras/layers/LSTM |
| `tf.keras.layers.Embedding` | tensorflow.org/api_docs/python/tf/keras/layers/Embedding |
| `tf.keras.layers.Dense` | tensorflow.org/api_docs/python/tf/keras/layers/Dense |
| Keras RNN guide | keras.io/guides/working_with_rnns/ |

### Recommended Reading Order

1. Colah's LSTM post — colah.github.io/posts/2015-08-Understanding-LSTMs/
2. TensorFlow text classification RNN tutorial — tensorflow.org/text/tutorials/text_classification_rnn
3. The Illustrated Transformer — jalammar.github.io/illustrated-transformer/
4. Attention Is All You Need — arxiv.org/abs/1706.03762